# Part 1: Standard KB with a Search Index knowledge source

In this notebook you will create a **search index**, populate it with sample documents,
wire it up as a **`searchIndex` knowledge source**, create a **knowledge base**, and
run a grounded **retrieve** call — all using the `2026-05-01-preview` REST API.

> **No SDK required** — every call is a plain `requests` HTTP call so you can see
> exactly what goes over the wire.

### Prerequisites

| Item | Details |
|------|---------|
| `.env` file | In the `notebooks/` directory (same folder as this notebook). Contains `BIGBUGS_ENDPOINT` and `BIGBUGS_API_KEY`. |
| Python packages | `requests`, `python-dotenv` (already in `requirements.txt`). |

> ⚠️ **The `.env` is currently in testing state — ask Matt for setup credentials.**
> Never check secrets into this repo.

## 1 — Setup: load environment and create a session

In [ ]:
import json, os, time, uuid
from pathlib import Path
from datetime import datetime, timezone

import requests

# ── Load secrets from .env in the notebook directory ─────────────────
ENV_PATH = Path(".").resolve() / ".env"

def load_env(path: Path) -> dict:
    values = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        values[k.strip()] = v.strip()
    return values

env = load_env(ENV_PATH)
ENDPOINT = env["BIGBUGS_ENDPOINT"].rstrip("/")
API_KEY  = env["BIGBUGS_API_KEY"]
API_VERSION = "2026-05-01-preview"

session = requests.Session()
session.headers.update({"api-key": API_KEY, "Accept": "application/json"})

def url(path: str) -> str:
    sep = "&" if "?" in path else "?"
    return f"{ENDPOINT}{path}{sep}api-version={API_VERSION}"

def pp(response: requests.Response):
    print(f"{response.status_code} {response.reason}")
    try:
        print(json.dumps(response.json(), indent=2))
    except Exception:
        print(response.text[:500])

stamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
INDEX_NAME = f"lab532-index-{stamp}"
KS_NAME    = f"lab532-searchindex-{stamp}"
KB_NAME    = f"lab532-searchkb-{stamp}"

print(f"Endpoint : {ENDPOINT}")
print(f"Index    : {INDEX_NAME}")
print(f"KS       : {KS_NAME}")
print(f"KB       : {KB_NAME}")

## 2 — Quick service health check

In [ ]:
r = session.get(url("/servicestats"))
pp(r)

## 3 — Create a search index with a semantic configuration

A `searchIndex` knowledge source requires its backing index to have at least
one **semantic configuration**.

In [ ]:
index_body = {
    "name": INDEX_NAME,
    "fields": [
        {"name": "id",       "type": "Edm.String", "key": True,  "retrievable": True},
        {"name": "title",    "type": "Edm.String", "searchable": True, "retrievable": True},
        {"name": "content",  "type": "Edm.String", "searchable": True, "retrievable": True},
        {"name": "category", "type": "Edm.String", "searchable": True, "filterable": True, "retrievable": True},
    ],
    "semantic": {
        "defaultConfiguration": "default-semantic",
        "configurations": [
            {
                "name": "default-semantic",
                "prioritizedFields": {
                    "titleField": {"fieldName": "title"},
                    "prioritizedContentFields": [{"fieldName": "content"}],
                },
            }
        ],
    },
}

r = session.put(url(f"/indexes('{INDEX_NAME}')"), json=index_body,
                headers={"Prefer": "return=representation"})
pp(r)

## 4 — Upload sample documents

In [ ]:
docs = {
    "value": [
        {
            "@search.action": "mergeOrUpload",
            "id": "1",
            "title": "Widget return policy",
            "content": "Widgets can be returned within 30 days of purchase with a receipt. "
                       "Refunds are issued to the original payment method within 5 business days.",
            "category": "policy",
        },
        {
            "@search.action": "mergeOrUpload",
            "id": "2",
            "title": "Widget installation guide",
            "content": "Install the widget by mounting the bracket on a flat surface "
                       "and tightening the four included screws. Refer to diagram A.",
            "category": "guide",
        },
        {
            "@search.action": "mergeOrUpload",
            "id": "3",
            "title": "Widget safety information",
            "content": "Keep widgets away from water and extreme heat. "
                       "Do not disassemble. Contact support if the unit is damaged.",
            "category": "safety",
        },
        {
            "@search.action": "mergeOrUpload",
            "id": "4",
            "title": "Employee health benefits overview",
            "content": "Full-time employees are eligible for medical, dental, and vision "
                       "coverage starting on day 1. Part-time employees are eligible after 90 days.",
            "category": "hr",
        },
        {
            "@search.action": "mergeOrUpload",
            "id": "5",
            "title": "Remote work policy",
            "content": "Employees may work remotely up to 3 days per week with manager approval. "
                       "A home office stipend of $500 is available annually.",
            "category": "hr",
        },
    ]
}

r = session.post(url(f"/indexes('{INDEX_NAME}')/docs/search.index"), json=docs)
pp(r)
print("\nWaiting 3s for indexing to settle...")
time.sleep(3)

## 5 — Create a `searchIndex` knowledge source

In [ ]:
ks_body = {
    "name": KS_NAME,
    "description": "Lab 532 searchIndex knowledge source",
    "kind": "searchIndex",
    "searchIndexParameters": {
        "searchIndexName": INDEX_NAME,
        "sourceDataFields": [
            {"name": "id"}, {"name": "title"}, {"name": "content"}, {"name": "category"}
        ],
        "searchFields": [{"name": "title"}, {"name": "content"}],
        "semanticConfigurationName": "default-semantic",
    },
}

r = session.put(url(f"/knowledgesources('{KS_NAME}')"), json=ks_body,
                headers={"Prefer": "return=representation"})
pp(r)

## 6 — Create a knowledge base referencing the source

In [ ]:
kb_body = {
    "name": KB_NAME,
    "description": "Lab 532 knowledge base — searchIndex backed",
    "outputMode": "extractiveData",
    "retrievalReasoningEffort": {"kind": "minimal"},
    "knowledgeSources": [{"name": KS_NAME}],
}

r = session.put(url(f"/knowledgebases('{KB_NAME}')"), json=kb_body,
                headers={"Prefer": "return=representation"})
pp(r)

## 7 — Retrieve: ask the KB a question

Key parameters:
- `maxOutputDocuments` controls how many grounding references come back.
- `includeActivity: true` shows which sources were consulted.
- `knowledgeSourceParams` lets you pass per-source overrides at query time.

In [ ]:
retrieve_body = {
    "intents": [{"type": "semantic", "search": "What is the widget return policy?"}],
    "retrievalReasoningEffort": {"kind": "minimal"},
    "includeActivity": True,
    "maxOutputDocuments": 3,
    "knowledgeSourceParams": [
        {
            "knowledgeSourceName": KS_NAME,
            "kind": "searchIndex",
            "includeReferences": True,
            "includeReferenceSourceData": True,
            "alwaysQuerySource": True,
        }
    ],
}

r = session.post(url(f"/knowledgebases('{KB_NAME}')/retrieve"), json=retrieve_body)
pp(r)

### Try a second query

In [ ]:
retrieve_body_2 = {
    "intents": [{"type": "semantic", "search": "What employee health benefits are available?"}],
    "retrievalReasoningEffort": {"kind": "minimal"},
    "includeActivity": True,
    "maxOutputDocuments": 2,
    "knowledgeSourceParams": [
        {
            "knowledgeSourceName": KS_NAME,
            "kind": "searchIndex",
            "includeReferences": True,
            "includeReferenceSourceData": True,
            "alwaysQuerySource": True,
        }
    ],
}

r = session.post(url(f"/knowledgebases('{KB_NAME}')/retrieve"), json=retrieve_body_2)
pp(r)

## 8 — Cleanup

Delete the KB, KS, and index in reverse dependency order.

In [ ]:
for label, path in [
    ("KB",    f"/knowledgebases('{KB_NAME}')"),
    ("KS",    f"/knowledgesources('{KS_NAME}')"),
    ("Index", f"/indexes('{INDEX_NAME}')"),
]:
    r = session.delete(url(path))
    print(f"Delete {label}: {r.status_code} {r.reason}")

## Known limitations on the current BigBugs stamp

| Feature | Status |
|---------|--------|
| `searchIndex` retrieve | ✅ Works |
| `maxOutputDocuments` (request-level) | ✅ Works — controls final output count |
| Per-source `maxOutputDocuments` | ⚠️ Accepted but no observable effect yet |
| KB-stored per-source retrieve defaults | ❌ Rejected at KB create time |
| `top` parameter on retrieve | ❌ Rejected — use `maxOutputDocuments` |
| `file` KS retrieve | ❌ Not supported on this stamp |

➡️ Continue to [Part 2: Add Fabric IQ as a first-class knowledge source type](part2-fabric-iq-to-kb.ipynb).